In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
GEMINI_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Your Ability

In [7]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.b-roll-autogenerator",
        description="Based on the inputted video create a prompt for b-roll footage",
        worker_release="qwen3-instruct",
        text_prompt="""
          You are an expert video editor and broadcast producer. Analyze the provided video footage to understand the core subject, tone, and narrative context.

          Based on your analysis, write a description of the ideal B-roll (supplemental footage) that should be edited over this scene to enhance the viewer's understanding and maintain visual interest.

          CRITICAL INSTRUCTIONS:

          Do NOT describe the current footage you are looking at.

          Only describe the new B-roll footage that needs to be sourced or shot.

          Describe the specific actions, environment, camera angles, and lighting of the B-roll.

          Your output must be exactly one paragraph. Do not include any conversational filler, markdown formatting, or introductions.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=550,
            image_size=512
        ),
        is_public=False
    )
]



### Create Your Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.describe.b-roll-autogenerator:latest"
   )
])

accumulated_descriptions = []

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
  endpoint.set_pop(pop)
  sample_vid_path = Path("/content/sample_video.mp4")
  job = endpoint.upload(sample_vid_path)

  while result := job.predict():
    texts = result.get("texts", [])
    for text_item in texts:
      raw_text = text_item.get("text", "")
      if raw_text:
        accumulated_descriptions.append(raw_text)

all_descriptions_string = "\n---\n".join(accumulated_descriptions)


In [ ]:
from google import genai

prompt = f"""
You are an expert video editor and prompt engineer for a video generation model.
I am providing you with multiple frame-by-frame AI descriptions of B-roll footage that should accompany a specific footage.

Because these are individual frame analyses from a video, the vision model generates slightly varying descriptions of the footage. Your primary job is to filter and determine the true "consensus" of what the B-roll should look like.

Please read through all the individual frame reports and synthesize them into ONE master video generation prompt based on these strict rules:

1. Find the Consensus: Identify the core subjects, actions, lighting, and cinematic styles that appear consistently across the majority of the frames.
2. Format for a Video Generator: Combine the consensus elements into a single, highly descriptive, comma-separated paragraph. Use strong visual keywords (e.g., "cinematic," "shallow depth of field," "4k," "slow motion").

Output ONLY the final master video generation prompt. Do not include introductory text, conversational filler, or formatting outside of the paragraph itself.

Here is the raw frame-by-frame data:
{all_descriptions_string}
"""

# Create the Gemini client and generate prompt
client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

# Print the result
print("\n=== B-ROLL GENERATION PROMPT ===")
print(response.text)